In [1]:
# Cell 1: Imports & Configuration
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import logging
from datetime import datetime

# Jupyter notebook configuration
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

from validation.create_embedding_plots import create_embedding_plots
from validation.data_loading import setup_data_with_error_handling

# LearnM8 imports
from learnm8.oracles import CSVOracle
from learnm8.core.data_manager import DataManager
from learnm8.learners.ensemble import RFEnsemble
from learnm8.acquisition import KennardStoneAcquisition
from learnm8.utils.data_loaders import load_benchmark_data

✅ Visualization function defined
✅ Data setup function defined


In [2]:
# Cell 2: Load Dataset
print("📊 Step 2: Loading dataset...")
target = "ADA"
target_path = "/home/tony/LearnM8/ESSENCE_benchmark_input/ADA.csv"
target_column = "ESSENCE-Dock_Score"

try:
    compound_pool, ground_truth = load_benchmark_data(str(target_path), target_column)
    print(f"Loaded {len(compound_pool)} compounds")
    print(f"Target column: {target_column}")
    print(f"Score range: {ground_truth[target_column].min():.2f} to {ground_truth[target_column].max():.2f}")
    
except Exception as e:
    print(f"Failed to load dataset: {e}")
    print(f"\n❌ Error loading dataset: {e}")

📊 Step 2: Loading dataset...
Found Activity column - enrichment calculations will be performed
Loaded 5543 compounds for benchmarking
Target column: ESSENCE-Dock_Score
Loaded 5543 compounds
Target column: ESSENCE-Dock_Score
Score range: -25.06 to -2.13


In [3]:
# Cell 3: DataManager Setup
print("⚙️  Step 3: Setting up data manager with error handling...")

# Initialize DataManager
data_manager = DataManager(results_dir=str(Path("./")), featurizer='morgan')

# Test and setup data with error handling
compound_pool = setup_data_with_error_handling(compound_pool, data_manager)

⚙️  Step 3: Setting up data manager with error handling...


In [4]:
# Cell 4: Initial Training
print("🎯 Step 4: Performing initial training...")

learner = RFEnsemble(n_estimators=3, random_states=[42, 43, 44])

# Determine initial training size
initial_size = min(100, len(compound_pool) // 10)  # Adaptive initial size
np.random.seed(42)
initial_indices = np.random.choice(len(compound_pool), size=initial_size, replace=False)

# Split data
labeled_compounds = compound_pool.iloc[initial_indices].copy()
unlabeled_compounds = compound_pool.drop(compound_pool.index[initial_indices]).copy()

# Create oracle and measure labeled compounds
oracle = CSVOracle(csv_path=str(target_path))
labeled_compounds = oracle.measure(labeled_compounds, [target_column])

# Train the model
learner.train(labeled_compounds, target_column, data_manager)
print(f"Initial training completed: {len(labeled_compounds)} labeled, {len(unlabeled_compounds)} unlabeled")

🎯 Step 4: Performing initial training...
Initial training completed: 100 labeled, 5443 unlabeled


In [ ]:
# Cell 5: KennardStone Acquisition
print("🎯 Step 5: Running KennardStone acquisition...")

# Set acquisition parameters
batch_fraction = 0.01
batch_size = max(1, int(batch_fraction * len(unlabeled_compounds)))
print(f"Batch size: {batch_size}")

print(f"\n🎯 Acquisition Parameters:")
print(f"  • Batch fraction: {batch_fraction*100:.1f}%")
print(f"  • Batch size: {batch_size:,}")
print(f"  • Unlabeled pool: {len(unlabeled_compounds):,}")

# Get predictions for unlabeled compounds
print("Getting predictions for unlabeled compounds...")
predictions, uncertainties = learner.predict(unlabeled_compounds, data_manager)
ground_truth_unlabeled = oracle.measure(unlabeled_compounds, [target_column])[target_column].values

# Prepare unlabeled compounds with predictions
unlabeled_with_predictions = unlabeled_compounds.copy()
unlabeled_with_predictions['prediction'] = predictions
if uncertainties is not None:
	unlabeled_with_predictions['uncertainty'] = uncertainties
else:
	unlabeled_with_predictions['uncertainty'] = np.zeros(len(predictions))

# Get KennardStone acquisition function
print("Initializing KennardStone acquisition function...")
init_start_time = datetime.now()
kennard_stone_acquisition = KennardStoneAcquisition(data_manager=data_manager,)
init_time = datetime.now() - init_start_time
print(f"Initialization time: {init_time.total_seconds():.2f} seconds")

# Perform acquisition
print("Performing KennardStone compound selection...")
selection_start_time = datetime.now()
selected_compounds = kennard_stone_acquisition.select(compounds=unlabeled_with_predictions, n_select=batch_size)
selected_indices = selected_compounds.index.values
selection_time = datetime.now() - selection_start_time
total_kennard_stone_time = init_time + selection_time
print(f"Selection time: {selection_time.total_seconds():.2f} seconds")

print(f"KennardStone selected {len(selected_indices)} compounds")

🎯 Step 5: Running KennardStone acquisition...
Batch size: 54

🎯 Acquisition Parameters:
  • Batch fraction: 1.0%
  • Batch size: 54
  • Unlabeled pool: 5,443
Getting predictions for unlabeled compounds...


Kennard-Stone selection failed: DataManager.__init__() missing 1 required positional argument: 'results_dir'


Initializing KennardStone acquisition function...
Initialization time: 0.00 seconds
Performing KennardStone compound selection...


RuntimeError: Kennard-Stone acquisition failed: DataManager.__init__() missing 1 required positional argument: 'results_dir'

In [ ]:
# Cell 6: Generate Embeddings & Visualizations
print("📊 Step 6: Generating visualizations...")

plots_dir = Path(f"./plots_{target}")

# Generate PCA embeddings from molecular fingerprints
print("Generating PCA embeddings from molecular fingerprints")
features = data_manager.get_features(
    compound_ids=unlabeled_compounds['ID'].tolist(),
    smiles_list=unlabeled_compounds['SMILES'].tolist(),
    featurizer_type='morgan'
)

# Generate PCA embeddings
pca = PCA(n_components=2, random_state=42)
features_array = features.toarray() if hasattr(features, 'toarray') else features
embeddings = pca.fit_transform(features_array)

# For KennardStone, we don't have cluster labels, so we'll use a single label
labels = np.zeros(len(unlabeled_compounds))  # All compounds in one group

# Convert selected indices to unlabeled dataset indices
unlabeled_positions = {idx: pos for pos, idx in enumerate(unlabeled_compounds.index)}
selected_positions = [unlabeled_positions[idx] for idx in selected_indices if idx in unlabeled_positions]

create_embedding_plots(embeddings, labels, selected_positions, f"KennardStone - {total_kennard_stone_time.total_seconds():.2f} seconds", plots_dir)

In [ ]:
import shutil
# Clean up temporary files
shutil.rmtree(Path(data_manager.results_dir / ".cache/"), ignore_errors=True)